In [5]:
!pip -q install flask pyngrok requests

In [2]:
from flask import Flask, request, jsonify
from decimal import Decimal, InvalidOperation
from datetime import datetime, timezone
from pathlib import Path
from getpass import getpass
import json
import logging
import secrets
import threading

app = Flask(__name__)

# Set a private webhook secret for this session.
WEBHOOK_SECRET = getpass("Create a webhook secret: ").strip()

if len(WEBHOOK_SECRET) < 16:
    raise ValueError("Use a secret with at least 16 characters.")

LOG_FILE = Path("/content/orders.jsonl")

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("order-webhook")


@app.get("/health")
def health():
    return jsonify({"status": "ok"}), 200


@app.post("/webhook/orders")
def receive_order():
    supplied_secret = request.headers.get("X-Webhook-Secret", "")

    if not secrets.compare_digest(supplied_secret, WEBHOOK_SECRET):
        return jsonify({"error": "Unauthorized"}), 401

    payload = request.get_json(silent=True)

    if not isinstance(payload, dict):
        return jsonify({"error": "Expected a JSON object"}), 400

    if not all(k in payload for k in ("order_id", "customer", "items")):
        return jsonify({"error": "Missing required fields"}), 400

    order_id = payload["order_id"]
    customer = payload["customer"]
    items = payload["items"]

    if not isinstance(order_id, (str, int)) or isinstance(order_id, bool):
        return jsonify({"error": "Invalid order_id"}), 400

    if not isinstance(customer, str) or not customer.strip():
        return jsonify({"error": "Invalid customer"}), 400

    if not isinstance(items, list) or not items:
        return jsonify({"error": "items must be a non-empty list"}), 400

    total = Decimal("0")
    clean_items = []

    for item in items:
        if not isinstance(item, dict):
            return jsonify({"error": "Each item must be an object"}), 400

        price = item.get("price")
        quantity = item.get("quantity")

        if (
            isinstance(price, bool)
            or not isinstance(price, (int, float))
            or not isinstance(quantity, int)
            or isinstance(quantity, bool)
            or quantity <= 0
        ):
            return jsonify({"error": "Invalid price or quantity"}), 400

        try:
            price_decimal = Decimal(str(price))
        except InvalidOperation:
            return jsonify({"error": "Invalid price"}), 400

        if not price_decimal.is_finite() or price_decimal < 0:
            return jsonify({"error": "Price must be finite and non-negative"}), 400

        total += price_decimal * quantity
        clean_items.append({
            "price": str(price_decimal),
            "quantity": quantity
        })

    event = {
        "received_at": datetime.now(timezone.utc).isoformat(),
        "order_id": str(order_id),
        "customer": customer.strip(),
        "item_count": len(clean_items),
        "order_total": str(total.quantize(Decimal("0.01"))),
        "status": "processed"
    }

    with LOG_FILE.open("a", encoding="utf-8") as f:
        f.write(json.dumps(event) + "\n")

    logger.info("Processed order %s", event["order_id"])
    return jsonify(event), 200


# Start Flask once in a background thread.
server_thread = threading.Thread(
    target=lambda: app.run(
        host="0.0.0.0",
        port=5000,
        debug=False,
        use_reloader=False
    ),
    daemon=True
)
server_thread.start()

print("Webhook server started on port 5000.")


Create a webhook secret: ··········
Webhook server started on port 5000.
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


In [10]:
from getpass import getpass
from pyngrok import ngrok

NGROK_TOKEN = getpass("Paste your ngrok authtoken: ").strip()

if not NGROK_TOKEN or any(c.isspace() for c in NGROK_TOKEN):
    raise ValueError("Token is empty or contains whitespace.")

ngrok.set_auth_token(NGROK_TOKEN)

tunnel = ngrok.connect(5000, "http")
PUBLIC_URL = tunnel.public_url

print("Webhook URL:", PUBLIC_URL)
print("Health check:", PUBLIC_URL + "/health")

Paste your ngrok authtoken: ··········
Webhook URL: https://outhouse-legend-molecular.ngrok-free.dev
Health check: https://outhouse-legend-molecular.ngrok-free.dev/health


In [11]:
import requests

payload = {
    "order_id": "ORD-1001",
    "customer": "Asha",
    "items": [
        {"name": "Headphones", "price": 1999, "quantity": 1},
        {"name": "Cable", "price": 299, "quantity": 2}
    ]
}

response = requests.post(
    PUBLIC_URL + "/webhook/orders",
    json=payload,
    headers={"X-Webhook-Secret": WEBHOOK_SECRET},
    timeout=30
)

print("HTTP status:", response.status_code)
print("Response:", response.json())

INFO:werkzeug:127.0.0.1 - - [17/Oct/2026 13:39:51] "POST /webhook/orders HTTP/1.1" 200 -


HTTP status: 200
Response: {'customer': 'Asha', 'item_count': 2, 'order_id': 'ORD-1001', 'order_total': '2597.00', 'received_at': '2026-10-17T13:39:51.518077+00:00', 'status': 'processed'}


In [12]:
# Test 1: Wrong secret
r1 = requests.post(
    PUBLIC_URL + "/webhook/orders",
    json=payload,
    headers={"X-Webhook-Secret": "wrong-secret"},
    timeout=30
)

print("Wrong secret:", r1.status_code)

INFO:werkzeug:127.0.0.1 - - [17/Oct/2026 13:40:11] "POST /webhook/orders HTTP/1.1" 401 -


Wrong secret: 401


In [13]:
# Test 2: Negative price
bad_payload = {
    "order_id": "ORD-1002",
    "customer": "Asha",
    "items": [
        {"name": "Headphones", "price": -100, "quantity": 1}
    ]
}

r2 = requests.post(
    PUBLIC_URL + "/webhook/orders",
    json=bad_payload,
    headers={"X-Webhook-Secret": WEBHOOK_SECRET},
    timeout=30
)

print("Invalid order:", r2.status_code)
print("Error:", r2.json())

INFO:werkzeug:127.0.0.1 - - [17/Oct/2026 13:40:28] "POST /webhook/orders HTTP/1.1" 400 -


Invalid order: 400
Error: {'error': 'Price must be finite and non-negative'}


In [14]:
from pathlib import Path

log_path = Path("/content/orders.jsonl")

if log_path.exists():
    for line in log_path.read_text(encoding="utf-8").splitlines():
        print(line)
else:
    print("No orders logged yet.")

{"received_at": "2026-10-17T13:39:51.518077+00:00", "order_id": "ORD-1001", "customer": "Asha", "item_count": 2, "order_total": "2597.00", "status": "processed"}


In [15]:
from google.colab import files

files.download("/content/orders.jsonl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>